In [ ]:
"""
avenue10_axis1_fano_evolutionary_automaton.py
====================================================================
AXIS 1: EVOLUTION AS ALGORITHMIC COMPRESSION (Rule FE)
Strict implementation of the Fano-Evolutionary Cellular Automaton.
====================================================================

By Néstor E. Ramos


"""
import numpy as np
import matplotlib.pyplot as plt
from math import gcd
from functools import reduce
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("AXIS 1: FANO-EVOLUTIONARY CELLULAR AUTOMATON (Rule FE)")
print("=" * 80)

# =============================================================================
# 1. FINITIST PARAMETERS
# =============================================================================
d_max = 9               # Modulo-9 alphabet (0-8)
L_genome = 10           # Genome length (Short algorithmic rule)
L_phenotype = 100       # Phenotype length (Target complexity)
#N_e = 5                 # Effective population size (Computational Buffer for Rule 2)
N_e = 18                # ¡Aumentado! Un búfer un poco mayor (ej. 18 o 27, múltiplos de 9)
                        # permite que la fricción aritmética deje "pasar" suficiente
                        # información para que la selección natural trabaje.
fano_period = 7         # Selection cycle (Rule 3)
generations = 200       # Evolutionary time
pop_size = 50           # Number of concurrent genomes

# Fano Weights (Fibonacci-ish, sum=17)
# w = [1, 2, 3, 5, 3, 2, 1]
fano_weights = np.array([1, 2, 3, 5, 3, 2, 1])

print(f"Alphabet: Modulo-{d_max}")
print(f"Genome Length: {L_genome} -> Phenotype Length: {L_phenotype}")
print(f"Buffer Size (N_e): {N_e} (Determines Arithmetic Friction)")

# =============================================================================
# 2. TARGET ENVIRONMENT (The "Resource" to be compressed)
# =============================================================================
# We create a target that is complex but generated by similar rules,
# ensuring a solution exists within the search space.
def generate_target():
    target = np.zeros(L_phenotype, dtype=int)
    # Create a pattern with period 9 (Modulo-9 structure)
    for i in range(L_phenotype):
        target[i] = (i * 3 + (i // 9)) % d_max
    return target

TARGET = generate_target()

# =============================================================================
# 3. RULE FE: THE AUTOMATON
# =============================================================================

def rule_fe_propagate(genome, steps=5, N_e=N_e):
    """
    Rule 1 & 2: Genetic State Propagation with Arithmetic Friction.
    Strict Modulo-9 implementation to prevent diffusion flattening.
    """
    # Inicializar repitiendo el genoma a lo largo del fenotipo
    grid = np.tile(genome, (L_phenotype // L_genome) + 1)[:L_phenotype].astype(float)

    for t in range(steps):
        new_grid = np.zeros_like(grid)
        for x in range(L_phenotype):
            # Obtener vecindario de 7 elementos (radio 3) con bordes periódicos
            neighbors = [grid[(x + i) % L_phenotype] for i in range(-3, 4)]

            # REGLA 1: Suma ponderada Fano
            weighted_sum = np.sum(fano_weights * np.array(neighbors))

            # Aplicamos el sustrato Módulo-9 DIRECTAMENTE para preservar la entropía
            val_modular = weighted_sum % d_max
            p_val = val_modular / (d_max - 1)  # Normalización limpia a [0, 1]

            # REGLA 2: Fricción Aritmética (Tratamiento de truncamiento en búfer finito N_e)
            p_quant = np.floor(p_val * N_e) / N_e

            # Mapeo de regreso al espacio discreto del alfabeto Módulo-9 (0-8)
            new_grid[x] = int(p_quant * (d_max - 1)) % d_max

        grid = new_grid
    return grid.astype(int)


def fano_consistency_check(genome):
    """
    Rule 3: Fano Consistency Check (Selection).
    Verifies topological compatibility.
    In Fano geometry, incidence relations must hold.
    We simulate this by checking if the genome sum satisfies a mod-7 constraint
    (Fano plane has 7 points/lines).
    """
    # Check if sum of genome is compatible with Fano modulus (7)
    # If not, it's "topologically incompatible"
    genome_sum = np.sum(genome)
    return (genome_sum % 7) == 0

def calculate_fitness(phenotype, target, genome_length):
    """
    Fitness = Compression Efficiency.
    High fitness = Low Error + Short Genome.
    """
    # Modulo-9 distance (circular distance)
    diff = np.abs(phenotype - target)
    diff = np.minimum(diff, d_max - diff) # Wrap around distance
    error = np.sum(diff)

    # Hardware cost
    hardware_cost = genome_length * 2.0

    # Compression Score (Inverse of cost)
    if error + hardware_cost == 0:
        return 1000.0
    return 100.0 / (error + hardware_cost)

# =============================================================================
# 4. EVOLUTIONARY LOOP
# =============================================================================

print("\nInitializing Population...")
# Population: List of genomes
population = [np.random.randint(0, d_max, size=L_genome) for _ in range(pop_size)]

best_fitness_history = []
avg_genome_length_history = [] # Will be constant here, but we track complexity via error
speciation_data = []

print("Running Fano-Evolutionary Automaton...")
for gen in range(generations):

    # Evaluate Fitness
    fitnesses = []
    phenotypes = []

    for genome in population:
        # Express Phenotype via Rule FE
        pheno = rule_fe_propagate(genome, steps=5, N_e=N_e)
        phenotypes.append(pheno)

        fit = calculate_fitness(pheno, TARGET, len(genome))

        # Rule 3: Fano Consistency Check (Every 7 ticks)
        if gen % fano_period == 0:
            if not fano_consistency_check(genome):
                # Failed check -> Arithmetic Scattering (Reduced Fitness / High Mutation risk)
                fit *= 0.1 # Penalize heavily

        fitnesses.append(fit)

    # Sort by fitness
    sorted_indices = np.argsort(fitnesses)[::-1]
    population = [population[i] for i in sorted_indices]
    phenotypes = [phenotypes[i] for i in sorted_indices]
    fitnesses = [fitnesses[i] for i in sorted_indices]

    best_fitness_history.append(fitnesses[0])

    # Selection & Reproduction (Memory Saturation)
    # Keep top 20%
    survivors = population[:int(pop_size * 0.4)]
    next_gen = []

    for _ in range(pop_size):
        # Select parent
        parent = survivors[np.random.randint(0, len(survivors))]
        child = parent.copy()

        # Mutation (Modulo-9 bit toggle)
        # Higher mutation if Fano check failed (simulated by scattering)
        mutation_rate = 0.3 if (gen % fano_period == 0 and not fano_consistency_check(parent)) else 0.1

        if np.random.rand() < mutation_rate:
            idx = np.random.randint(0, L_genome)
            delta = np.random.choice([-1, 1])
            child[idx] = (child[idx] + delta) % d_max

        next_gen.append(child)

    population = next_gen

# =============================================================================
# 5. SPECIATION ANALYSIS (Rule 4)
# =============================================================================
print("\nAnalyzing Speciation (Topological Confinement)...")
# Take final population
final_pop = np.array(population)
# Calculate pairwise GCD of differences to find species clusters
# Simplified: Cluster by sum of genome mod 9
species_labels = np.sum(final_pop, axis=1) % d_max
unique_species, counts = np.unique(species_labels, return_counts=True)

# =============================================================================
# 6. VISUALIZATION
# =============================================================================

print("Generating Visualizations...")
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Axis 1: Fano-Evolutionary Automaton (Rule FE)', fontsize=16, fontweight='bold')

# Panel 1: Fitness Dynamics
ax1 = fig.add_subplot(2, 2, 1)
# Reemplaza la línea de ploteo del Panel 1 por esta para ver el fitness real en los checkpoints:
generaciones_fano = np.arange(0, generations, fano_period)
fitness_fano = [best_fitness_history[i] for i in generaciones_fano]

ax1.plot(best_fitness_history, 'r-', alpha=0.3, label='Instant Fitness')
ax1.plot(generaciones_fano, fitness_fano, 'k-o', linewidth=2, label='Structural Evolution (Fano Checkpoints)')

#ax1.plot(best_fitness_history, 'r-', linewidth=2, label='Best Compression Efficiency')
ax1.set_title('1. Fitness Dynamics\n(Compression Efficiency vs. Generations)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Generation'); ax1.set_ylabel('Fitness Score (1/Cost)')
# Mark Fano periods
for t in range(0, generations, fano_period):
    ax1.axvline(x=t, color='gray', alpha=0.2, linestyle='--')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Panel 2: Phenotype Compression (Target vs Best)
ax2 = fig.add_subplot(2, 2, 2)
best_pheno = phenotypes[0] # Best phenotype from last generation
x_axis = np.arange(L_phenotype)

ax2.plot(x_axis, TARGET, 'k--', linewidth=2, label='Environmental Target')
ax2.plot(x_axis, best_pheno, 'b-', linewidth=1.5, label=f'Best Phenotype (Rule FE Output)')
ax2.set_title('2. Phenotype Compression\n(Target vs. Best Organism Expression)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Position (x)'); ax2.set_ylabel('Modulo-9 State')
ax2.legend(); ax2.grid(True, alpha=0.3)

# Panel 3: Arithmetic Friction Mechanism (Rule 2 Visualization)
ax3 = fig.add_subplot(2, 2, 3)
p_continuous = np.linspace(0, 1, 1000)
p_quant = np.floor(p_continuous * N_e) / N_e
ax3.plot(p_continuous, p_quant, 'g-', linewidth=2, label=f'Quantization (N_e={N_e})')
ax3.plot([0, 1], [0, 1], 'k:', linewidth=1, label='Continuous (No Friction)')
ax3.set_title('3. Arithmetic Friction (Rule 2)\n(Deterministic Truncation)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Continuous Value (p)'); ax3.set_ylabel('Quantized Value')
ax3.legend(); ax3.grid(True, alpha=0.3)

# Panel 4: Speciation & Physical Interpretation
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')

textstr = (
    "PHYSICAL INTERPRETATION (Rule FE):\n\n"
    "1. Rule 1 (Propagation):\n"
    "   Genotypes evolve via Fano-weighted neighborhood sum (mod 9).\n"
    "   Weights [1,2,3,5,3,2,1] embed Fibonacci scaling.\n\n"
    "2. Rule 2 (Arithmetic Friction):\n"
    "   The 'Round' function quantizes states to steps of 1/N_e.\n"
    "   This replaces random drift with deterministic information loss.\n\n"
    "3. Rule 3 (Fano Selection):\n"
    "   Every 7 ticks, genotypes must satisfy mod-7 consistency.\n"
    "   Failure triggers arithmetic scattering (reduced fitness).\n\n"
    "4. Rule 4 (Speciation):\n"
    f"   Final population split into {len(unique_species)} distinct topological species\n"
    "   based on Modulo-9 arithmetic incompatibility (GCD > 1)."
)

props = dict(boxstyle='round', facecolor='wheat', alpha=0.95, edgecolor='black')
ax4.text(0.05, 0.95, textstr, transform=ax4.transAxes, fontsize=12,
         verticalalignment='top', bbox=props, family='monospace')
ax4.set_title('4. The Finitist Resolution', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('axis1_fano_evolutionary_automaton.png', dpi=1200, bbox_inches='tight')
print("✓ Saved: axis1_fano_evolutionary_automaton.png")
plt.show()

# =============================================================================
# 7. VERDICT
# =============================================================================

print("\n" + "=" * 80)
print("SIMULATION VERDICT: FANO-EVOLUTIONARY AUTOMATON")
print("=" * 80)
print(f"Final Best Fitness: {best_fitness_history[-1]:.4f}")
print(f"Number of Species (Topological Clusters): {len(unique_species)}")
print(f"Species Distribution: {dict(zip(unique_species, counts))}")
print("\nKey Finitist Insights:")
print("• Evolution is driven by Fano-weighted propagation (Rule 1).")
print("• Drift is deterministic quantization error (Rule 2).")
print("• Selection is topological consistency checking (Rule 3).")
print("• Speciation is arithmetic incompatibility (Rule 4).")
print("=" * 80)

AXIS 1: FANO-EVOLUTIONARY CELLULAR AUTOMATON (Rule FE)
Alphabet: Modulo-9
Genome Length: 10 -> Phenotype Length: 100
Buffer Size (N_e): 18 (Determines Arithmetic Friction)

Initializing Population...
Running Fano-Evolutionary Automaton...
